# Lesson 2 · Advanced Exploratory Data Analysis · Starter notebook

Module 8 · Information Systems · Inteli · Prof. Afonso Brandão

This notebook supports the flipped format of Lesson 2. Before the session, each member runs the first part of the notebook on synthetic data that the notebook itself generates, and arrives able to explain what Steps 1 and 2 return. During the session, the group completes Steps 3 to 8 on the partner data.

**Before running.** Complete the environment checklist of the reading material (Python 3.12, Java 17 or later, Poetry and Jupyter) and open the notebook inside the project environment:

```
poetry run jupyter notebook
```

The data is fictitious. It reproduces the pharmacy chain used as the illustrative case of Lessons 1 and 2, with the tables `store`, `product`, `sale_line` and `stock_position`, and it deliberately contains the defects that the procedure is designed to find. No figure produced here describes the partner.

## 0. Generate the synthetic extract

The cell below creates four CSV files in `./data/` with a fixed random seed, so that every member obtains the same data. The later cells read these files in the same way they would read a real extract.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
Path("data").mkdir(exist_ok=True)

# store
store = pd.DataFrame({
    "store_id": range(1, 7),
    "city": ["Campinas", "Santos", "Sorocaba", "Jundiaí", "Bauru", "Franca"],
    "region": ["Interior", "Coast", "Interior", "Interior", "Interior", "Interior"],
})

# product: about 40% are continuous-use medicines
product = pd.DataFrame({
    "product_id": range(1, 41),
    "name": [f"Product {i:02d}" for i in range(1, 41)],
    "continuous_use": rng.random(40) < 0.4,
})

# stock_position: declared grain is one row per product, store and day
dates = pd.date_range("2026-08-01", "2026-09-30", freq="D")
stock = (pd.MultiIndex.from_product(
            [store.store_id, product.product_id, dates],
            names=["store_id", "product_id", "date"])
         .to_frame(index=False))
stock["on_hand"] = rng.poisson(12, len(stock)).astype(float)
stock.loc[rng.random(len(stock)) < 0.06, "on_hand"] = 0           # stock-outs
missing = (stock.store_id == 4) & (rng.random(len(stock)) < 0.25)  # one store without daily counts
stock.loc[missing, "on_hand"] = np.nan
negative = stock.sample(6, random_state=7).index                   # invariant violations
stock.loc[negative, "on_hand"] = -rng.integers(1, 4, len(negative))
dups = stock.sample(frac=0.02, random_state=11).copy()             # about 2% of keys sent twice
resent = rng.random(len(dups)) < 0.5
dups.loc[resent, "on_hand"] = dups.loc[resent, "on_hand"] + 1      # half of them with a different count
stock = pd.concat([stock, dups], ignore_index=True).sample(frac=1, random_state=3)

# sale_line: a few orphan product_ids and a few unusually large quantities
n = 20_000
sale_line = pd.DataFrame({
    "sale_id": range(1, n + 1),
    "product_id": rng.integers(1, 41, n),
    "store_id": rng.integers(1, 7, n),
    "quantity": rng.integers(1, 4, n),
    "sold_at": dates[0] + pd.to_timedelta(rng.integers(0, len(dates) * 86_400, n), unit="s"),
})
sale_line.loc[sale_line.sample(40, random_state=5).index, "product_id"] = rng.choice([41, 42], 40)
sale_line.loc[sale_line.sample(5, random_state=9).index, "quantity"] = [150, 180, 220, 260, 300]

store.to_csv("data/store.csv", index=False)
product.to_csv("data/product.csv", index=False)
stock.to_csv("data/stock_position.csv", index=False)
sale_line.to_csv("data/sale_line.csv", index=False)
print("Synthetic extract written to ./data/")

## Step 1 · Start from the canvas

The analysis of a table begins by stating why the table is explored. The question comes from block 2 of the Data Model Canvas, the grain from the declaration of the first cube, and the invariants from the information viewpoint written in Lesson 1.

- **Table:** `stock_position`
- **Analytical question (block 2):** which stores run out of continuous-use medicines most often, and of which products?
- **Declared grain:** one row per product, store and day.
- **Invariants to check:** `on_hand >= 0`; every `(store_id, product_id)` exists in `store` and `product`; every `product_id` of `sale_line` exists in `product`.

## Step 2 · Load and inventory

Dates are parsed at load time, so that the period covered is computed from real dates. Jupyter shows only the last expression of a cell, so each result is wrapped in `display`.

In [ ]:
sales = pd.read_csv("data/sale_line.csv", parse_dates=["sold_at"])
stock = pd.read_csv("data/stock_position.csv", parse_dates=["date"])
products = pd.read_csv("data/product.csv")
stores = pd.read_csv("data/store.csv")

display(stock.shape)
display(stock.dtypes)
display(stock.memory_usage(deep=True).sum() / 1e6)   # megabytes
display(stock["date"].agg(["min", "max"]))          # period covered
display(sales.shape)
display(sales["sold_at"].agg(["min", "max"]))

---
# In class · Steps 3 to 8

The cells below are completed by the group during the session, first on this synthetic data and then on the partner data. Each step has a short instruction and an empty cell. Section numbers refer to the reading material of Lesson 2.

## Step 3 · Structure and grain

Test the declared key `["store_id", "product_id", "date"]` with `duplicated(subset=...)`, count the repeated keys and display the repeated rows side by side. Decide whether they are identical resends or a finer grain (material §8).

In [ ]:
# TODO

## Step 4 · Completeness

Compute the null rate of each column and break the rate of `on_hand` down by store and by week. A rate concentrated in one store is a process failure to report, and it is not imputed (material §8).

In [ ]:
# TODO

## Step 5 · Univariate profile

Describe `quantity`, compute the cardinality of each column and count the outliers with the 1.5 × IQR rule. Record the volume and examples of the outliers (material §9).

In [ ]:
# TODO

## Step 6 · Relationships between variables

Plot daily sales, compute the first draft of the stock-out share by store with the deduplicated and non-null `obs` table, and compute a Spearman correlation on measures only, never on identifiers (material §10).

In [ ]:
# TODO

## Step 7 · Consistency across tables

Measure the share of orphan sale lines with `merge(..., indicator=True)` and count the violations of each invariant of Step 1 (material §11).

In [ ]:
# TODO

## Step 8 · Record and decide

For each finding of Steps 3 to 7, add one row to the data quality log below, with its volume, probable cause, decision and canvas block (material §12).

In [ ]:
# TODO

## Data quality log

Copy this table into the repository, under `/docs/data_model_canvas/`, and fill one row per finding.

| ID | Table | Check | Finding and volume | Probable cause | Decision | Canvas block |
|----|-------|-------|--------------------|----------------|----------|--------------|
| DQ-01 |  |  |  |  |  |  |

## Idempotent Parquet write

Write the explored table to the raw zone, partitioned by ingestion date, with `existing_data_behavior="delete_matching"` and a fixed `basename_template`, so that running the cell twice leaves the same rows. Read it back and compare the types and the row count with Step 2 (material §15).

In [ ]:
# TODO